# FESOMP Diagnostics Tutorial

This notebook demonstrates how to use the `fesomp.diag` module for computing oceanographic diagnostics from FESOM2 model output.

**Diagnostics covered:**
- Sea ice area, volume, and extent
- Hovmöller diagrams (depth-time plots)
- Volume-weighted means
- Ocean heat content
- Mixed layer depth

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import fesomp

%matplotlib inline

## 1. Load Mesh and Data

First, let's load the FESOM2 mesh and some example data.

In [ ]:
# Load mesh
mesh = fesomp.load_mesh('/Users/nkolduno/PYTHON/DATA/CORE27_mesh/fesom.mesh.diag.nc')
print(mesh)
print(f"\nMesh has {mesh.nlev} vertical levels ({mesh.nlev-1} layers)")
print(f"Nodes: {mesh.n2d}, Elements: {mesh.nelem}")
print(f"Depth range: {mesh.depth_levels[0]:.0f}m to {mesh.depth_levels[-1]:.0f}m")

In [ ]:
# Load 3D temperature data
temp_ds = xr.open_dataset('/Users/nkolduno/PYTHON/DATA/CORE27_data/temp.fesom.1958.nc')
print(temp_ds)

In [ ]:
# Extract temperature with time dimension
temp = temp_ds['temp']  # shape: (time, nz1, n2d)
print(f"Temperature shape: {temp.shape}")
print(f"Dimensions: {temp.dims}")

In [ ]:
# Load sea ice data
sic_ds = xr.open_dataset('/Users/nkolduno/PYTHON/DATA/CORE27_data/a_ice.fesom.1958.nc')
siv_ds = xr.open_dataset('/Users/nkolduno/PYTHON/DATA/CORE27_data/m_ice.fesom.1958.nc')

sic = sic_ds['a_ice']  # Sea ice concentration
siv = siv_ds['m_ice']  # Sea ice effective thickness (volume per area)

print(f"Sea ice concentration shape: {sic.shape}")
print(f"Sea ice volume shape: {siv.shape}")

## 2. Sea Ice Diagnostics

The `fesomp.diag` module provides functions for computing sea ice area, volume, and extent.

In [ ]:
# Get node areas from mesh geometry
node_area = mesh.geometry.node_area[0]  # Surface areas
print(f"Node area shape: {node_area.shape}")
print(f"Total surface area: {node_area.sum()/1e12:.2f} million km²")

### 2.1 Sea Ice Area

Sea ice area is the sum of (ice concentration × cell area) - the actual area covered by ice.

In [ ]:
# Compute sea ice area for both hemispheres
nh_ice_area = fesomp.diag.ice_area(sic, node_area, mesh.lat, hemisphere="N")
sh_ice_area = fesomp.diag.ice_area(sic, node_area, mesh.lat, hemisphere="S")

print(f"NH ice area: {type(nh_ice_area)}")
print(f"NH ice area range: {float(nh_ice_area.min())/1e12:.2f} - {float(nh_ice_area.max())/1e12:.2f} million km²")
print(f"SH ice area range: {float(sh_ice_area.min())/1e12:.2f} - {float(sh_ice_area.max())/1e12:.2f} million km²")

In [ ]:
# Plot seasonal cycle
fig, ax = plt.subplots(figsize=(12, 5))

time_vals = np.arange(len(nh_ice_area))
ax.plot(time_vals, nh_ice_area.values / 1e12, 'b-o', label='Northern Hemisphere', linewidth=2)
ax.plot(time_vals, sh_ice_area.values / 1e12, 'r-o', label='Southern Hemisphere', linewidth=2)

ax.set_xlabel('Time step')
ax.set_ylabel('Sea Ice Area (million km²)')
ax.set_title('Sea Ice Area Time Series')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 2.2 Sea Ice Extent

Sea ice extent is the total area of cells where ice concentration exceeds a threshold (typically 15%).

In [ ]:
# Compute ice extent with 15% threshold
nh_ice_extent = fesomp.diag.ice_extent(sic, node_area, mesh.lat, hemisphere="N", threshold=0.15)
sh_ice_extent = fesomp.diag.ice_extent(sic, node_area, mesh.lat, hemisphere="S", threshold=0.15)

# Plot comparison: area vs extent
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(time_vals, nh_ice_area.values / 1e12, 'b-o', label='Ice Area', linewidth=2)
ax.plot(time_vals, nh_ice_extent.values / 1e12, 'b--s', label='Ice Extent (>15%)', linewidth=2)
ax.set_xlabel('Time step')
ax.set_ylabel('Area (million km²)')
ax.set_title('Northern Hemisphere')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(time_vals, sh_ice_area.values / 1e12, 'r-o', label='Ice Area', linewidth=2)
ax.plot(time_vals, sh_ice_extent.values / 1e12, 'r--s', label='Ice Extent (>15%)', linewidth=2)
ax.set_xlabel('Time step')
ax.set_ylabel('Area (million km²)')
ax.set_title('Southern Hemisphere')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.3 Sea Ice Volume

In [ ]:
# Compute ice volume
nh_ice_vol = fesomp.diag.ice_volume(siv, node_area, mesh.lat, hemisphere="N")
sh_ice_vol = fesomp.diag.ice_volume(siv, node_area, mesh.lat, hemisphere="S")

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(time_vals, nh_ice_vol.values / 1e12, 'b-o', label='Northern Hemisphere', linewidth=2)
ax.plot(time_vals, sh_ice_vol.values / 1e12, 'r-o', label='Southern Hemisphere', linewidth=2)
ax.set_xlabel('Time step')
ax.set_ylabel('Sea Ice Volume (10³ km³)')
ax.set_title('Sea Ice Volume Time Series')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot sea ice concentration map for first time step
fig, ax, _ = fesomp.plot(
    sic[0].values,
    mesh.lon,
    mesh.lat,
    mapproj="npstere",
    titles="Sea Ice Concentration (Arctic)",
    units="fraction",
    cmap="Blues",
    levels=(0, 1, 20),
    coastlines=True,
)
plt.show()

## 3. Vertical Diagnostics

### 3.1 Hovmöller Diagram

A Hovmöller diagram shows the area-weighted mean profile over time.

In [ ]:
# Compute global mean temperature profile for each time step
temp_np = temp.values  # (time, nlev-1, n2d)
print(f"Temperature data shape: {temp_np.shape}")

# Compute Hovmöller data
hovm = fesomp.diag.hovmoller(temp_np, mesh.geometry.node_area)
print(f"Hovmöller data shape: {hovm.shape}")

In [ ]:
# Plot Hovmöller diagram
time_values = np.arange(temp_np.shape[0])
depth_values = mesh.depth_layers

fig, ax = plt.subplots(figsize=(14, 6))
pcm = ax.pcolormesh(time_values, depth_values, hovm.T, 
                     shading='auto', cmap='RdYlBu_r')
ax.set_ylim(2000, 0)  # Invert y-axis for depth
ax.set_xlabel('Time step')
ax.set_ylabel('Depth (m)')
ax.set_title('Global Mean Temperature Profile (Hovmöller Diagram)')
plt.colorbar(pcm, ax=ax, label='Temperature (°C)')
plt.tight_layout()
plt.show()

### 3.2 Volume-Weighted Mean

Compute the volume-weighted mean temperature over a specific depth range.

In [ ]:
# Compute mean temperature in different depth ranges
depth_ranges = [(0, 100), (0, 700), (0, 2000), (700, 2000)]

print("Volume-weighted mean temperature by depth range:")
print("-" * 50)

for depth_range in depth_ranges:
    mean_temp = fesomp.diag.volume_mean(
        temp_np,
        node_area,
        mesh.depth_levels,
        depth_range=depth_range
    )
    print(f"{depth_range[0]:4d}-{depth_range[1]:4d}m: {mean_temp.mean():.2f}°C (time mean)")

In [ ]:
# Time series of volume-weighted mean temperature
mean_0_700 = fesomp.diag.volume_mean(temp_np, node_area, mesh.depth_levels, depth_range=(0, 700))
mean_700_2000 = fesomp.diag.volume_mean(temp_np, node_area, mesh.depth_levels, depth_range=(700, 2000))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(time_values, mean_0_700, 'r-o', label='0-700m', linewidth=2)
ax.plot(time_values, mean_700_2000, 'b-o', label='700-2000m', linewidth=2)
ax.set_xlabel('Time step')
ax.set_ylabel('Temperature (°C)')
ax.set_title('Volume-Weighted Mean Temperature')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3.3 Ocean Heat Content

Ocean heat content (OHC) is computed as: OHC = ρ × cp × (T - T_ref) × Volume

In [ ]:
# Compute ocean heat content in top 700m (standard metric)
ohc_700 = fesomp.diag.heat_content(
    temp_np,
    node_area,
    mesh.depth_levels,
    depth_range=(0, 700),
    reference_temp=0.0  # Reference temperature
)

# And in top 2000m
ohc_2000 = fesomp.diag.heat_content(
    temp_np,
    node_area,
    mesh.depth_levels,
    depth_range=(0, 2000),
    reference_temp=0.0
)

print(f"OHC 0-700m: {ohc_700.mean()/1e22:.2f} × 10²² J")
print(f"OHC 0-2000m: {ohc_2000.mean()/1e22:.2f} × 10²² J")

In [ ]:
# Plot OHC time series
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(time_values, ohc_700 / 1e22, 'r-o', label='0-700m', linewidth=2)
ax.plot(time_values, ohc_2000 / 1e22, 'b-o', label='0-2000m', linewidth=2)
ax.set_xlabel('Time step')
ax.set_ylabel('Ocean Heat Content (10²² J)')
ax.set_title('Global Ocean Heat Content')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Mixed Layer Depth

Mixed layer depth (MLD) can be computed using temperature or density thresholds.

In [ ]:
# Compute MLD using temperature threshold (0.2°C)
# Using the first time step
temp_t0 = temp_np[0]  # (nlev-1, n2d)
print(f"Temperature profile shape: {temp_t0.shape}")

mld = fesomp.diag.mixed_layer_depth(
    temp_t0,
    mesh.depth_layers,  # Use layer centers for layer data
    threshold=0.2,
    criterion="temperature",
    reference_depth=10.0
)

print(f"MLD shape: {mld.shape}")
print(f"MLD range: {mld.min():.0f}m - {mld.max():.0f}m")
print(f"Mean MLD: {mld.mean():.0f}m")

In [ ]:
# Plot MLD map
fig, ax, _ = fesomp.plot(
    mld,
    mesh.lon,
    mesh.lat,
    mapproj="robin",
    titles="Mixed Layer Depth (T criterion, 0.2°C threshold)",
    units="m",
    cmap="viridis_r",
    levels=(0, 500, 25),
    coastlines=True,
)
plt.show()

In [ ]:
# Compare different threshold values
thresholds = [0.1, 0.2, 0.5, 1.0]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, thresh in zip(axes, thresholds):
    mld_t = fesomp.diag.mixed_layer_depth(
        temp_t0,
        mesh.depth_layers,
        threshold=thresh,
        criterion="temperature"
    )
    
    _, _, _ = fesomp.plot(
        mld_t,
        mesh.lon,
        mesh.lat,
        ax=[ax],
        mapproj="robin",
        titles=f"MLD (threshold = {thresh}°C)",
        units="m",
        cmap="viridis_r",
        levels=(0, 500, 25),
        coastlines=True,
    )

plt.tight_layout()
plt.show()

## 5. Regional Analysis

You can apply regional masks to compute diagnostics for specific areas.

In [ ]:
# Create a simple regional mask (e.g., North Atlantic)
north_atlantic_mask = (
    (mesh.lon >= -80) & (mesh.lon <= 0) &
    (mesh.lat >= 0) & (mesh.lat <= 65)
)

print(f"North Atlantic nodes: {north_atlantic_mask.sum()} out of {mesh.n2d}")

In [ ]:
# Compute heat content for North Atlantic only
ohc_na_700 = fesomp.diag.heat_content(
    temp_np,
    node_area,
    mesh.depth_levels,
    depth_range=(0, 700),
    mask=north_atlantic_mask
)

print(f"North Atlantic OHC 0-700m: {ohc_na_700.mean()/1e22:.3f} × 10²² J")

In [ ]:
# Compute volume-weighted mean for North Atlantic
na_mean_temp = fesomp.diag.volume_mean(
    temp_np,
    node_area,
    mesh.depth_levels,
    depth_range=(0, 700),
    mask=north_atlantic_mask
)

global_mean_temp = fesomp.diag.volume_mean(
    temp_np,
    node_area,
    mesh.depth_levels,
    depth_range=(0, 700)
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(time_values, global_mean_temp, 'b-o', label='Global', linewidth=2)
ax.plot(time_values, na_mean_temp, 'r-o', label='North Atlantic', linewidth=2)
ax.set_xlabel('Time step')
ax.set_ylabel('Temperature (°C)')
ax.set_title('Volume-Weighted Mean Temperature (0-700m)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Summary

### Available Diagnostic Functions

**Sea Ice:**
- `fesomp.diag.ice_area(sic, node_area, lat, hemisphere)` - Sea ice area
- `fesomp.diag.ice_volume(siv, node_area, lat, hemisphere)` - Sea ice volume
- `fesomp.diag.ice_extent(sic, node_area, lat, hemisphere, threshold)` - Sea ice extent

**Vertical:**
- `fesomp.diag.hovmoller(data, node_area)` - Area-weighted mean profile
- `fesomp.diag.volume_mean(data, node_area, depth_levels, depth_range)` - Volume-weighted mean
- `fesomp.diag.heat_content(temp, node_area, depth_levels, depth_range)` - Ocean heat content
- `fesomp.diag.total_volume(node_area, depth_levels, depth_range)` - Total volume

**Mixed Layer:**
- `fesomp.diag.mixed_layer_depth(data, depth, threshold, criterion)` - MLD calculation
- `fesomp.diag.mixed_layer_depth_interpolated(...)` - MLD with interpolation

### Key Features

- Works with both numpy arrays and xarray DataArrays
- Supports dask for parallel computation
- Optional regional masking for all functions
- Flexible depth range selection

In [ ]:
# Clean up
plt.close('all')
print("\nDiagnostics tutorial complete!")